# Spotlight Your Instructions: Instruction-following with Dynamic Attention Steering

This Apple Silicon notebook demonstrates **Spotlight** with the Metal backend. It uses `HookLLMMetal` and the Metal Spotlight worker while preserving the same `generate_with_spotlight()` API as the CUDA/local notebook.

**Paper**: [Venakteswaran and Contractor, EACL 2026](https://aclanthology.org/2026.eacl-long.174/)


### Installation

Use the `vllm-metal` environment described in `notebooks/metal/README.md`, then install this repo into that environment before opening the notebook.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "metal":
    repo_root = repo_root.parents[1]
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

plugin_src = repo_root / "vllm_hook_plugins"
if str(plugin_src) not in sys.path:
    sys.path.insert(0, str(plugin_src))

print(f"Repo root: {repo_root}")
print(f"Plugin source: {plugin_src}")


### Imports & Environment


In [ ]:
import os
import multiprocessing as mp

import torch
from vllm import SamplingParams
from vllm_hook_plugins.metal import HookLLMMetal
from vllm_hook_plugins import generate_with_spotlight

os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")
os.environ.setdefault("VLLM_METAL_USE_PAGED_ATTENTION", "0")
os.environ.setdefault("VLLM_METAL_MEMORY_FRACTION", "auto")

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

print("Metal Spotlight environment configured")


### Initialize `HookLLMMetal`


In [ ]:
cache_dir = str(repo_root / "cache")
model = "Qwen/Qwen2-1.5B-Instruct"

llm = HookLLMMetal(
    model=model,
    worker_name="probe_spotlight",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=0.35,
    max_model_len=2048,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
)

print(f"Model loaded: {model}")
print("Metal Spotlight worker enabled")


### Configure Test Parameters


In [ ]:
prompt = (
    "Return the response for the following as a JSON: "
    "Write a 400 word paragraph on France and its food specifically "
    "focussing on dishes. Include the paragraph and the list of dishes "
    "mentioned the paragraph in seperate fields."
)
emph_strings = ["Return the response for the following as a JSON:"]
alpha = 0.1
temperature = 0.0
max_tokens = 500

sampling_params = SamplingParams(temperature=temperature, max_tokens=max_tokens)


### Generate Baseline


In [ ]:
outputs_baseline = llm.generate(
    prompts=[prompt],
    sampling_params=sampling_params,
    use_hook=False,
)
baseline_text = outputs_baseline[0].outputs[0].text
print(baseline_text)


### Generate With Spotlight


In [ ]:
outputs_spotlight = generate_with_spotlight(
    llm,
    prompts=[prompt],
    emph_strings=emph_strings,
    alpha=alpha,
    sampling_params=sampling_params,
)
spotlight_text = outputs_spotlight[0].outputs[0].text
print(spotlight_text)


### Comparison


In [ ]:
print("=" * 70)
print("COMPARISON")
print("=" * 70)
print(f"Prompt: {prompt}")
print(f"Emphasized span(s): {emph_strings}")
print(f"Alpha: {alpha}")
print("\nBASELINE")
print("-" * 70)
print(baseline_text)
print("\nWITH SPOTLIGHT")
print("-" * 70)
print(spotlight_text)
